# NotebookLM Kit — Notebook Inventory

Read-only view of a notebook's contents.

1. **Setup** — authenticate and verify `tsx`
2. **Config** — set your notebook ID
3. **Sources** — list all sources (indexed)
4. **Artifacts** — list all artifacts with their type, creation time, and the indices of the sources used to generate each one (referencing the table above)

In [100]:
import importlib, pipeline, pipeline._core, pipeline._sources, pipeline._artifacts, pipeline.config
importlib.reload(pipeline.config)
importlib.reload(pipeline._core)
importlib.reload(pipeline._sources)
importlib.reload(pipeline._artifacts)
importlib.reload(pipeline)

from pipeline import load_credentials, check_tsx

creds = load_credentials(mode="auto")
check_tsx()


credentials: using patchright (credentials.json found)
Credentials ready — token: 42 chars, cookies: 2316 chars
tsx: tsx v4.22.3
node v20.17.0


## Config

Find your notebook ID in the NotebookLM URL:  
`https://notebooklm.google.com/notebook/` **`<YOUR_NOTEBOOK_ID>`**

In [101]:
NOTEBOOK_ID = "64d7d18f-a2db-4b0b-959d-5276506dffad"  # ← paste your notebook ID here

## Sources

`list_sources(notebook_id, creds)` — prints an indexed table of all sources.  
The index column is what the **Artifacts** table below uses to reference each source.

In [102]:
from pipeline import list_sources

sources = list_sources(NOTEBOOK_ID, creds)


Notebook : Mastering Modern Data Orchestration with Dagster
+---+--------------------------------------------------------------------------+------------+--------------------------------------+
| # | Title                                                                    | Status     | Source ID                            |
+---+--------------------------------------------------------------------------+------------+--------------------------------------+
| 0 | A Quick Introduction to Machine Learning with Dagster | HackerNoon       | READY      | fbcb2f70-5f48-4a30-93ce-6099e82a0cd5 |
| 1 | Best Practices for Machine Learning with Dagster                         | READY      | e06a6437-95a0-4bb0-b886-66fcb4e3b57a |
| 2 | Dagster for Data Orchestration: Architecture & Capabilities - Atlan      | READY      | 7b94d750-50a8-4d6c-a059-9fdc8d2bd502 |
| 3 | Dagster training: Orchestrating data pipelines in the cloud - Ambient IT | READY      | bf05939f-1b17-44f5-a64e-6a2a83e5d514 |
| 4 | Da

## Artifacts

`list_artifacts(notebook_id, sources, creds)` — every artifact in the notebook with:

- **Title** and **Type** (FLASHCARDS, QUIZ, VIDEO, …)
- **Created** — local time pulled from the artifact's own `createdAt`
- **Sources** — comma-separated **indices** into the sources table above (e.g. `0,2` means rows 0 and 2)

In [103]:
from pipeline import list_artifacts

artifacts = list_artifacts(NOTEBOOK_ID, sources, creds)


Artifacts in notebook 64d7d18f-a2db-4b0b-959d-5276506dffad (8 total)
+----+----------------------------------------------+-------+---------------------+---------------------+--------------------------------------+
| #  | Title                                        | Type  | Created             | Sources             | Artifact ID                          |
+----+----------------------------------------------+-------+---------------------+---------------------+--------------------------------------+
|  0 | ML Quiz                                      | QUIZ  | 2026-05-31 07:40:15 | 1                   | 40622509-0c85-4f33-ad06-8f5265713538 |
|  1 | Dagster Quiz                                 | QUIZ  | 2026-05-30 20:34:58 | 1                   | 0d253666-7a46-492d-ad0b-82f64ea98a15 |
|  2 | Best Practices for Machine Lea 260530 1033   | QUIZ  | 2026-05-30 10:33:36 | 1                   | 46537b9b-bf0d-4521-8573-cbe5d1a1ddf2 |
|  3 | Best Practices for Machine Lea 260530 0920   | QUIZ  

## Rename Single-Source Artifacts

`rename_single_source_artifacts(artifacts, sources, creds, *, indices=None, dry_run=False)`

For every artifact whose `sourceIds` contains exactly **one** source, rename it to:

```
<source title> YYMMDD HHMM
```

where the timestamp comes from the artifact's own `createdAt` (local time).

- `indices` — optional list of row numbers from the Artifacts table above to limit the operation; `None` means all single-source artifacts.
- `dry_run=True` — preview the planned renames without calling the API.

Multi-source artifacts (e.g. audio overviews built from all sources) are left untouched.

In [106]:
from pipeline import rename_single_source_artifacts

# Preview only — no changes made
rename_single_source_artifacts(artifacts, sources, creds, dry_run=False)

# Limit to specific rows from the Artifacts table:
# rename_single_source_artifacts(artifacts, sources, creds, indices=[0, 1, 3], dry_run=True)


Renaming 6 single-source artifact(s)
+----+--------------------------------------------+----------------------------------------------------------------+----------+
| #  | Old title                                  | New title                                                      | Status   |
+----+--------------------------------------------+----------------------------------------------------------------+----------+
|  0 | ML Quiz                                    | Best Practices for Machine Learning with Dagster 260531 074015 | ok       |
|  1 | Dagster Quiz                               | Best Practices for Machine Learning with Dagster 260530 203458 | ok       |
|  2 | Best Practices for Machine Lea 260530 1033 | Best Practices for Machine Learning with Dagster 260530 103336 | ok       |
|  3 | Best Practices for Machine Lea 260530 0920 | Best Practices for Machine Learning with Dagster 260530 092023 | ok       |
|  4 | Best Practices for Machine Lea 260530 0907 | Best Practices

[{'index': 0,
  'artifactId': '40622509-0c85-4f33-ad06-8f5265713538',
  'oldTitle': 'ML Quiz',
  'newTitle': 'Best Practices for Machine Learning with Dagster 260531 074015',
  'status': 'ok',
  'error': None},
 {'index': 1,
  'artifactId': '0d253666-7a46-492d-ad0b-82f64ea98a15',
  'oldTitle': 'Dagster Quiz',
  'newTitle': 'Best Practices for Machine Learning with Dagster 260530 203458',
  'status': 'ok',
  'error': None},
 {'index': 2,
  'artifactId': '46537b9b-bf0d-4521-8573-cbe5d1a1ddf2',
  'oldTitle': 'Best Practices for Machine Lea 260530 1033',
  'newTitle': 'Best Practices for Machine Learning with Dagster 260530 103336',
  'status': 'ok',
  'error': None},
 {'index': 3,
  'artifactId': '38e69a5f-a414-4790-a99a-dff7ab621789',
  'oldTitle': 'Best Practices for Machine Lea 260530 0920',
  'newTitle': 'Best Practices for Machine Learning with Dagster 260530 092023',
  'status': 'ok',
  'error': None},
 {'index': 4,
  'artifactId': 'e5302356-b2c7-4e6a-b8de-096242f3bb91',
  'oldTitle

## Download Artifacts by Type

download_artifacts_by_type(artifacts, artifact_type, notebook_id, creds, *, indices=None, output_dir=None)

Downloads every artifact of the given type from the rtifacts list (or a subset via indices).
Files land in outputs/<artifact_type_lower>/<Notebook Name>/<yyyymmdd_hhmmss>_<title><ext>.
Already-downloaded files are skipped automatically.

Valid types: FLASHCARDS, QUIZ, AUDIO, INFOGRAPHIC, VIDEO, SLIDE_DECK.

In [107]:
from pipeline import download_artifacts_by_type

download_artifacts_by_type(artifacts, "Audio", NOTEBOOK_ID, creds)

# Limit to specific rows from the Artifacts table:
# download_artifacts_by_type(artifacts, "VIDEO", NOTEBOOK_ID, creds, indices=[0, 2, 5])
# download_artifacts_by_type(artifacts[2:6], "AUDIO", NOTEBOOK_ID, creds)


Downloaded 2 / 2 artifact(s)  →  d:\Core\_Code D\notebooklm-kit\outputs\audio\Mastering Modern Data Orchestration with Dagster
+---+----------------------------------------------+----------------------------------------------+------------+
| # | Source                                       | File                                         | Status     |
+---+----------------------------------------------+----------------------------------------------+------------+
| 0 | Replacing Airflow Tasks With Dagster Assets  | Replacing Airflow Tasks With Dagster Asse... | downloaded |
| 1 | Dagster Orchestrates Assets Instead Of Tasks | Dagster Orchestrates Assets Instead Of Ta... | downloaded |
+---+----------------------------------------------+----------------------------------------------+------------+


[{'sourceTitle': 'Replacing Airflow Tasks With Dagster Assets',
  'notebookTitle': 'Mastering Modern Data Orchestration with Dagster',
  'createdAt': '2026-02-26T04:44:25.868Z',
  'filePath': 'd:\\Core\\_Code D\\notebooklm-kit\\outputs\\audio\\Mastering Modern Data Orchestration with Dagster\\Replacing Airflow Tasks With Dagster Assets.mp3',
  'status': 'downloaded'},
 {'sourceTitle': 'Dagster Orchestrates Assets Instead Of Tasks',
  'notebookTitle': 'Mastering Modern Data Orchestration with Dagster',
  'createdAt': '2026-02-26T04:44:17.779Z',
  'filePath': 'd:\\Core\\_Code D\\notebooklm-kit\\outputs\\audio\\Mastering Modern Data Orchestration with Dagster\\Dagster Orchestrates Assets Instead Of Tasks.mp3',
  'status': 'downloaded'}]